# Example: Yield Calculation for a Zero
In this example, we will calculate the annualized investment yield of a zero-coupon bond given its price, face value, discount rate, and time to maturity. We'll explore how this yield relates to the discount rate used in pricing the instrument.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Compute zero-coupon investment yield:__ Calculate an annualized investment yield from price, face value, and time to maturity under a stated convention. State the day-count and annualization choices that the calculation depends on.
> * __Relate price and yield:__ Explain why the yield implied by a fixed maturity payment changes inversely with the security's market price. Recompute the yield when the purchase price or the remaining time to maturity changes.
> * __Distinguish rate concepts:__ Separate the rate used in a valuation model from a quoted investment yield and a realized holding-period return. Convert between a simple annualized yield and a compounded discount rate under a stated compounding convention.


We will calculate and interpret the yield of a zero-coupon security.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
# Load the notebook environment and local pricing helpers.
include(joinpath(@__DIR__, "Include.jl"));

  Activating 

project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
We are going to look at a hypothetical 52-week U.S. Treasury Bill. The data for this security is encoded in [a `NamedTuple` data structure](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple), which we'll call `bill_data::NamedTuple`:

In [2]:
# Define the zero-coupon bill and its quoted annual yield.
bill_data = (
    T = "52-Week" |> securityterm,
    n = 2, # classroom compounding periods per year
    par = 100.0,
    y = 0.04538, # 4.538% nominal annual yield
    c̄ = 0.0, # zero-coupon contract
);

Let's build a `zero_coupon_model::MyUSTreasuryZeroCouponBondModel` instance using the `bill_data::NamedTuple` data structure.

In [3]:
# Build and price the bill under the discrete-compounding convention.
zero_coupon_model = let

    # The discount model populates the price and discount-factor fields.
    discount_model = DiscreteCompoundingModel();
    zero_coupon_model = nothing;

    # Map the NamedTuple fields to the zero-coupon model inputs.
    zero_coupon_model = build(MyUSTreasuryZeroCouponBondModel, (
        par = bill_data.par,
        n = bill_data.n, 
        T = bill_data.T,
        rate =  bill_data.y,
    )) |> discount_model;

    zero_coupon_model # Return the priced bill model.
end;

What is the price of this bill at auction? 

In [4]:
# Display the model price in USD per 100 USD of face value.
zero_coupon_model.price

95.62366236777791

### Constants
We set some constant values that we use in the calculations below. Please see the comment next to each constant for its meaning, permissible units, etc.

In [5]:
# Retain the original 364-day maturity for the yield-conversion check below.
t = 364.0; # days to maturity at auction (52 weeks of 7 days)

___

## Task 1: Computing the Annualized Investment Yield at Auction and Afterward
In this task, we define the annualized investment yield of a zero-coupon instrument and compute it for a purchase made in the secondary market. The __annualized investment yield__ of a zero-coupon instrument with $t$ days to maturity (units: days), face (par) value $V_{P}$ (units: USD), and purchase price $V_{B,t}$ (units: USD) is given by:
$$
\boxed{
\begin{aligned}
Y(t) & = \underbrace{\left(\frac{V_{P}}{V_{B,t}} - 1\right)}_{\text{HPR}} \times \underbrace{\left(\frac{1}{T}\right)}_{\text{annualize}}\quad\\
\end{aligned}}
$$
where $\text{HPR}$ is the __holding period return__, $T = t/365$ is the time to maturity (units: years), and the $1/T$ term is an __annualization factor__.

> __Key Insight:__ At auction, the simple annualized investment yield is mathematically related to the compounded yield $y$ used in our pricing model through the relationship shown in Task 2. Later, the yield to remaining maturity must be recomputed from the current secondary-market price and remaining time. The original purchase yield remains a historical transaction measure; it is not the instrument's current market yield. Day-count and compounding conventions must be stated when comparing quoted yields.

Suppose we buy the bill on the secondary market with 182 days remaining, at a price 2.50 USD above the model auction price. The code below stores the annualized yield in `Y::Float64` and the holding-period return in `HPR::Float64`.


In [6]:
# Compute the holding-period return and its simple annualized investment yield.
Y,HPR = let

    # Treat the secondary-market purchase as occurring with 182 days remaining.
    t = 182.0; # days to maturity
    T = t/365.0; # actual/365 maturity in years
    V_P = zero_coupon_model.par; # face value in USD
    V_B = zero_coupon_model.price; # model auction price in USD
    δ = 2.5; # secondary-market price premium above the model auction price

    HPR  = V_P/(V_B + δ) - 1; # return from the secondary purchase price to face value
    Y    = HPR / T; # simple annualization over the remaining maturity

    (Y,HPR) # Return annualized yield and holding-period return.
end

(0.03834941257868965, 0.01912217284745621)

### Discussion
Let's consider a few scenarios to see what happens when we change the current price and remaining time to maturity of our example Treasury bill. Each answer below is a yield to remaining maturity under the stated simple annualization convention, not the investor's realized six-month holding-period return.
1. Suppose we purchased the bill at auction and, six months later, 182 days remain to maturity. If its current secondary-market price happens to equal the original auction price, what is its annualized yield to remaining maturity? Why can it differ from the original yield even though the dollar price is unchanged?
2. With 182 days remaining, suppose the current price is 1 USD above the auction price. What is the annualized yield to remaining maturity?
3. With 182 days remaining, suppose the current price is 1 USD below the auction price. What is the annualized yield to remaining maturity?

___

## Task 2: How Does the Investment Yield Relate to the Discount Rate?
In this task, we will explore how the annualized investment yield at auction relates to the compounded discount rate used in pricing the zero-coupon instrument. The relationship between the annualized investment yield $Y$ and the compounded discount rate $y$ is given by:
$$
\boxed{
\begin{aligned}
y & = n\left[\left(1 + Y\;T\right)^{1/(nT)} - 1\right]
\end{aligned}}
$$

> __Technical Note:__ This formula converts the simple annualized investment yield $Y$ back to the compounded discount rate $y$ used in pricing. This relationship allows us to verify that our yield calculations are consistent with the original pricing parameters.

Let's substitute the auction values and see what we observe. The code below recomputes the investment yield at auction and stores the converted discount rate in `r̂::Float64`.

In [7]:
# Convert the auction investment yield to an equivalent nominal compounded yield.
r̂ = let

    # Use the original auction horizon stored in the global t value.
    T = t/365.0; # actual/365 maturity in years
    V_P = zero_coupon_model.par; # face value in USD
    V_B = zero_coupon_model.price; # model auction price in USD
    n = zero_coupon_model.n; # compounding periods per year

    # Recompute the simple annualized investment yield at auction.
    Y_auction = (V_P/V_B - 1)/T; # holding-period return over the full horizon, annualized

    # Match the simple horizon return with an n-times-compounded annual quote.
    n*((1 + Y_auction*T)^(1/(n*T)) - 1) # Return the converted discount rate.
end

0.0453800000000002

Let's check: Is the converted discount rate the same as the original discount rate used to price the instrument at auction?

In [8]:
# Verify that the converted yield recovers the rate used to price the bill.
@assert isapprox(r̂, zero_coupon_model.rate; atol=1e-6)

___

## Summary

This example computed the annualized investment yield of a zero-coupon security and compared that quoted yield with related discount-rate and holding-period concepts.

> __Key Takeaways:__
>
> * __Investment yield annualizes the price-to-face-value gain:__ The purchase price, fixed maturity payment, remaining time, and stated annualization convention determine the quoted yield. Changing any one of these inputs changes the quoted number.
> * __Price and implied yield move inversely:__ With face value and maturity fixed, paying a higher price leaves a smaller gain to maturity and therefore a lower yield. Paying a lower price leaves a larger gain and a higher yield.
> * __Rate labels are not interchangeable:__ A valuation discount rate, a quoted investment yield, and a realized holding-period return answer different questions and can use different conventions. Always check the convention before comparing two quoted rates.

A yield calculation is interpretable only after its cash flows, time horizon, annualization rule, and compounding convention have been specified.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.